# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore, extract, and process a Croissant dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema JSON-LD file, which makes it self-describing and reproducible.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset schema and display dataset information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
meta = dataset.metadata
print(f"Dataset: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}\n")
print(f"Keywords: {getattr(meta, 'keywords', [])}\n")
print(f"Number of record sets: {len(meta.record_sets)}")

## 2. Data Overview
Review available record sets, their fields, and relevant `@id` for data extraction and analysis. All references to dataset entities use the `@id` from the Croissant schema.

In [ ]:
# List all record sets and their fields with @id
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet: {record_set.name} - @id: {record_set.id}")
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

## 3. Data Extraction
Load all available record sets into DataFrames for analysis. Use the `@id` values for record sets and fields listed above.

In [ ]:
# Extract record set @id values
record_set_ids = [rec_set.id for rec_set in dataset.metadata.record_sets]
dataframes = {}

for rec_id in record_set_ids:
    # Iterating and loading all records into a DataFrame for each record set
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set: {rec_id}")
    if not dataframes[rec_id].empty:
        print(f"Columns: {dataframes[rec_id].columns.tolist()}")
    print()

# Pick the first record set as an example
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Preview of first few records from '{main_rs_id}':")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering and normalizing a numeric field. All fields are referenced by their `@id`.

In [ ]:
# Choose the main record set and a numeric field for demonstration
# Review the printed field list above to choose a numeric field @id -- update as appropriate

selected_record_set = main_rs_id

# Guess a numeric field -- use first Float/Integer field
fields = next(rs for rs in dataset.metadata.record_sets if rs.id==selected_record_set).fields
numeric_field = None
for f in fields:
    if f.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number']:
        numeric_field = f.id
        print(f"Using numeric field: {f.name} (@id: {f.id}) with data type {f.data_type}")
        break

if numeric_field is None:
    print("No numeric field found for EDA.")
else:
    df = dataframes[selected_record_set]
    # Make sure to handle missing/NA values
    df = df.dropna(subset=[numeric_field])
    # Try converting to numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    df = df.dropna(subset=[numeric_field])

    # Set threshold for filtering
    threshold = df[numeric_field].quantile(0.75)  # top 25% as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)}")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field from the same record set (if possible)
    group_field = None
    for f in fields:
        if f.data_type in ['Text', 'schema:Text', 'Category', 'String'] and f.id != numeric_field:
            group_field = f.id
            print(f"Grouping by: {f.name} (@id: {f.id})")
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution, and optionally its grouped means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field before/after filtering
if numeric_field is not None and not df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set {selected_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=15, color='orange')
    plt.title(f"Distribution of {numeric_field} (Filtered, Top 25%)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group, if such exists
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field} (Filtered)")
        plt.xticks(rotation=60)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- Using `mlcroissant`, you loaded a Croissant dataset directly from its schema URL and explored its programmatic structure.
- You examined available record sets and fields using `@id` for robust referencing as recommended for FAIR data.
- Typical data analysis steps (filtering by value, normalizing, grouping, and plotting distributions) were demonstrated.

You can further customize this notebook based on the scientific needs, or automate batch processing of new FAIR datasets defined by Croissant schema with the same approach!